# Notebook 4: Gold Layer - Business Aggregates

**Objective**: Create aggregated views ready for analysis and reporting.

## What is the Gold Layer?

The **Gold Layer** is the presentation layer of the lakehouse:
- **Aggregates**: Pre-computed data for performance
- **KPIs**: Key performance indicators
- **Optimized for reporting**: Fast queries for dashboards

## Data Flow

```
Silver (clean) ──► Aggregation ──► Gold (business ready)
```

---
## INITIALIZATION

In [1]:
# Configure path to project
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark
from pyspark.sql.functions import sum as spark_sum, round as spark_round, count

# Create Spark session
spark = get_spark("gold-aggregations")

# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("=" * 60)
print("GOLD LAYER - BUSINESS AGGREGATES")
print("=" * 60)

GOLD LAYER - BUSINESS AGGREGATES


## 1. Read Silver Table

The Gold layer reads from Silver (already cleaned data).

In [2]:
# Read Silver table
sales_silver = spark.table("nessie.silver.sales")

silver_count = sales_silver.count()
print(f"Records in Silver: {silver_count:,}")

# Preview
print("\n=== Silver Preview ===")
sales_silver.select("order_id", "category", "region", "sales", "profit").show(5, truncate=False)

Records in Silver: 9,917

=== Silver Preview ===
+--------------+---------------+------+-------+--------+
|order_id      |category       |region|sales  |profit  |
+--------------+---------------+------+-------+--------+
|CA-2019-100006|Technology     |East  |377.97 |109.6113|
|CA-2019-100090|Furniture      |West  |502.488|-87.9354|
|CA-2019-100293|Office Supplies|South |91.056 |31.8696 |
|CA-2019-100328|Office Supplies|East  |3.928  |1.3257  |
|CA-2019-100363|Office Supplies|West  |2.368  |0.8288  |
+--------------+---------------+------+-------+--------+
only showing top 5 rows



## 2. Create Main Aggregate: Sales by Category and Region

**KPIs calculated**:
- `total_sales`: Sum of sales
- `total_profit`: Sum of profits
- `total_quantity`: Total quantity sold
- `order_count`: Number of orders

This aggregate allows analyzing performance by Category × Region combination.

In [3]:
# Aggregate by category and region
sales_by_category_region = (
    sales_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

print("=== Aggregate: Sales by Category × Region ===")
sales_by_category_region.orderBy("total_sales", ascending=False).show(truncate=False)

=== Aggregate: Sales by Category × Region ===
+---------------+-------+-----------+------------+--------------+-----------+
|category       |region |total_sales|total_profit|total_quantity|order_count|
+---------------+-------+-----------+------------+--------------+-----------+
|Technology     |East   |260324.31  |45688.43    |1912          |528        |
|Furniture      |West   |252284.98  |11413.18    |2682          |706        |
|Technology     |West   |250685.68  |44186.07    |2311          |595        |
|Office Supplies|West   |220154.61  |52473.51    |7147          |1883       |
|Furniture      |East   |206859.67  |3235.16     |2192          |595        |
|Office Supplies|East   |204234.56  |40691.37    |6388          |1698       |
|Technology     |Central|169800.88  |33430.62    |1530          |418        |
|Office Supplies|Central|166378.41  |8705.09     |5360          |1415       |
|Furniture      |Central|163412.11  |-2820.54    |1818          |479        |
|Technology     |S

## 3. Other Useful Aggregates

In [4]:
# Aggregate: Sales by Segment (Consumer, Corporate, Home Office)
from pyspark.sql.functions import avg as spark_avg

sales_by_segment = (
    sales_silver
    .groupBy("segment")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_round(spark_avg("sales"), 2).alias("avg_order_value"),
        count("order_id").alias("order_count")
    )
)

print("=== Aggregate: Sales by Segment ===")
sales_by_segment.orderBy("total_sales", ascending=False).show(truncate=False)

=== Aggregate: Sales by Segment ===
+-----------+-----------+------------+---------------+-----------+
|segment    |total_sales|total_profit|avg_order_value|order_count|
+-----------+-----------+------------+---------------+-----------+
|Consumer   |1147645.62 |133770.3    |222.84         |5150       |
|Corporate  |704431.87  |91745.7     |234.58         |3003       |
|Home Office|427405.42  |59678.13    |242.29         |1764       |
+-----------+-----------+------------+---------------+-----------+



In [5]:
# Aggregate: Top products
top_products = (
    sales_silver
    .groupBy("category", "sub_category", "product_name")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity")
    )
)

print("=== Top 10 Products ===")
top_products.orderBy("total_sales", ascending=False).show(10, truncate=False)

=== Top 10 Products ===
+---------------+------------+---------------------------------------------------------------------------+-----------+------------+--------------+
|category       |sub_category|product_name                                                               |total_sales|total_profit|total_quantity|
+---------------+------------+---------------------------------------------------------------------------+-----------+------------+--------------+
|Technology     |Copiers     |Canon imageCLASS 2200 Advanced Copier                                      |61599.82   |25199.93    |20            |
|Office Supplies|Binders     |Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind|27453.38   |7753.04     |31            |
|Technology     |Machines    |Cisco TelePresence System EX90 Videoconferencing Unit                      |22638.48   |-1811.08    |6             |
|Furniture      |Chairs      |HON 5400 Series Task Chairs for Big and Tall                    

## 4. Drop Old Gold Tables (if they exist)

In [6]:
# Cleanup: drop tables if they exist
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_segment")
spark.sql("DROP TABLE IF EXISTS nessie.gold.top_products")

print("Existing tables dropped")

Existing tables dropped


## 5. Create Gold Tables

In [7]:
# Create Gold tables
sales_by_category_region.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()
sales_by_segment.writeTo("nessie.gold.sales_by_segment").using("iceberg").create()
top_products.writeTo("nessie.gold.top_products").using("iceberg").create()

print("Gold tables created:")
print("- nessie.gold.sales_by_category_region")
print("- nessie.gold.sales_by_segment")
print("- nessie.gold.top_products")

Gold tables created:
- nessie.gold.sales_by_category_region
- nessie.gold.sales_by_segment
- nessie.gold.top_products


## 6. Verify Gold Tables

In [8]:
# List all Gold tables
print("=== Tables in Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

=== Tables in Gold ===
+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|gold     |sales_by_category_region|false      |
|gold     |sales_by_segment        |false      |
|gold     |top_products            |false      |
+---------+------------------------+-----------+



In [9]:
# Verify records
print("=== Record Counts ===")
spark.sql("SELECT 'sales_by_category_region' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_category_region").show()
spark.sql("SELECT 'sales_by_segment' as table_name, COUNT(*) as count FROM nessie.gold.sales_by_segment").show()
spark.sql("SELECT 'top_products' as table_name, COUNT(*) as count FROM nessie.gold.top_products").show()

=== Record Counts ===
+--------------------+-----+
|          table_name|count|
+--------------------+-----+
|sales_by_category...|   12|
+--------------------+-----+

+----------------+-----+
|      table_name|count|
+----------------+-----+
|sales_by_segment|    3|
+----------------+-----+

+------------+-----+
|  table_name|count|
+------------+-----+
|top_products| 1850|
+------------+-----+



## 7. Analytical Query Examples

Gold tables are optimized for business queries.

In [10]:
# Top 5 by sales
print("=== Top 5 Category × Region ===")
spark.sql("""
    SELECT category, region, total_sales, total_profit, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

=== Top 5 Category × Region ===
+---------------+------+-----------+------------+-----------+
|category       |region|total_sales|total_profit|order_count|
+---------------+------+-----------+------------+-----------+
|Technology     |East  |260324.31  |45688.43    |528        |
|Furniture      |West  |252284.98  |11413.18    |706        |
|Technology     |West  |250685.68  |44186.07    |595        |
|Office Supplies|West  |220154.61  |52473.51    |1883       |
|Furniture      |East  |206859.67  |3235.16     |595        |
+---------------+------+-----------+------------+-----------+



In [11]:
# Best margin by region
print("=== Profit Margin by Region ===")
spark.sql("""
    SELECT 
        region,
        SUM(total_sales) as sales,
        SUM(total_profit) as profit,
        ROUND(100.0 * SUM(total_profit) / SUM(total_sales), 2) as margin_pct
    FROM nessie.gold.sales_by_category_region
    GROUP BY region
    ORDER BY profit DESC
""").show(truncate=False)

=== Profit Margin by Region ===
+-------+---------+------------------+----------+
|region |sales    |profit            |margin_pct|
+-------+---------+------------------+----------+
|West   |723125.27|108072.76000000001|14.95     |
|East   |671418.54|89614.95999999999 |13.35     |
|South  |385347.7 |48191.24          |12.51     |
|Central|499591.4 |39315.170000000006|7.87      |
+-------+---------+------------------+----------+



In [12]:
# Performance by category
print("=== Performance by Category ===")
spark.sql("""
    SELECT 
        category,
        SUM(total_sales) as global_sales,
        SUM(total_profit) as global_profit,
        SUM(order_count) as global_orders
    FROM nessie.gold.sales_by_category_region
    GROUP BY category
    ORDER BY global_sales DESC
""").show(truncate=False)

=== Performance by Category ===
+---------------+-----------------+-------------+-------------+
|category       |global_sales     |global_profit|global_orders|
+---------------+-----------------+-------------+-------------+
|Technology     |828694.05        |143130.31    |1832         |
|Furniture      |734770.97        |20343.75     |2105         |
|Office Supplies|716017.8900000001|121720.07    |5980         |
+---------------+-----------------+-------------+-------------+



In [13]:
# Performance by segment
print("=== Performance by Segment ===")
spark.sql("""
    SELECT 
        segment,
        total_sales,
        total_profit,
        avg_order_value,
        order_count
    FROM nessie.gold.sales_by_segment
    ORDER BY total_sales DESC
""").show(truncate=False)

=== Performance by Segment ===
+-----------+-----------+------------+---------------+-----------+
|segment    |total_sales|total_profit|avg_order_value|order_count|
+-----------+-----------+------------+---------------+-----------+
|Consumer   |1147645.62 |133770.3    |222.84         |5150       |
|Corporate  |704431.87  |91745.7     |234.58         |3003       |
|Home Office|427405.42  |59678.13    |242.29         |1764       |
+-----------+-----------+------------+---------------+-----------+



## COMPLETE LAKEHOUSE SUMMARY

In [14]:
# Get counts from each layer
bronze_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.gold.sales_by_category_region").first()[0]

print("""
╔══════════════════════════════════════════════════════════╗
║              COMPLETE LAKEHOUSE - SUMMARY                ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Medallion Architecture:                                 ║
║                                                          ║
║  ┌─────────┐     ┌─────────┐     ┌─────────────┐         ║
║  │ BRONZE  │────►│ SILVER  │────►│    GOLD     │         ║
║  │  Raw    │     │  Clean  │     │ Aggregates  │         ║
║  └─────────┘     └─────────┘     └─────────────┘         ║
║    {:>6,}          {:>6,}           {:>6,}               ║
║                                                          ║
║  Tables created:                                         ║
║    Bronze:  nessie.bronze.sales                          ║
║    Silver:  nessie.silver.sales                          ║
║    Gold:    nessie.gold.sales_by_category_region         ║
║             nessie.gold.sales_by_segment                 ║
║             nessie.gold.top_products                     ║
║                                                          ║
║  Technologies:                                           ║
║    • Apache Spark (processing)                           ║
║    • Apache Iceberg (ACID table format)                  ║
║    • Project Nessie (versioned catalog)                  ║
║    • AWS S3 (storage)                                    ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""".format(bronze_count, silver_count, gold_count))

print("Bronze → Silver → Gold pipeline completed successfully!")
print("Next step: Run '06_complete_demo.ipynb' for Nessie demo")


╔══════════════════════════════════════════════════════════╗
║              COMPLETE LAKEHOUSE - SUMMARY                ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  Medallion Architecture:                                 ║
║                                                          ║
║  ┌─────────┐     ┌─────────┐     ┌─────────────┐         ║
║  │ BRONZE  │────►│ SILVER  │────►│    GOLD     │         ║
║  │  Raw    │     │  Clean  │     │ Aggregates  │         ║
║  └─────────┘     └─────────┘     └─────────────┘         ║
║    10,365           9,917               12               ║
║                                                          ║
║  Tables created:                                         ║
║    Bronze:  nessie.bronze.sales                          ║
║    Silver:  nessie.silver.sales                          ║
║    Gold:    nessie.gold.sales_by_category_region         ║
║             nessie.go